# Retail Orders — Executable Data Profile
Dataset: `retail-orders-raw.csv`  |  Dictionary: `retail-data-dictionary.csv`

This notebook runs **executable checks** across the five data-quality dimensions defined in the
Data Quality Contract:

1. **Completeness** — are required fields populated?
2. **Uniqueness** — is `order_id` unique?
3. **Validity** — do values conform to type/range/domain rules from the dictionary?
4. **Consistency** — do categorical fields normalize to one canonical value set?
5. **Freshness** — how current is the most recent valid `order_date`?

Every check below prints a **PASS/FAIL** verdict against the thresholds in the contract, plus the
offending `order_id`s so issues can be traced back to source rows.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RAW_PATH = 'retail-orders-raw.csv'
DICT_PATH = 'retail-data-dictionary.csv'

raw = pd.read_csv(RAW_PATH, dtype=str)  # read everything as string first — we control typing ourselves
data_dict = pd.read_csv(DICT_PATH)

print(f"Rows: {len(raw)}, Columns: {len(raw.columns)}")
raw


Rows: 12, Columns: 9


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
5,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0,Failed
6,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10,Paid
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105,Paid
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0,Pending
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0,Paid


## 0. Reference: Data Dictionary (quality rules as authored)

In [2]:
data_dict

,column,data_type,business_definition,quality_rule
0,order_id,string,Unique order identifier,Required and unique
1,order_date,date,Date the customer placed the order,Required ISO date between 2025-01-01 and today
2,customer_segment,category,Commercial customer segment,Student Fresher or Professional after normaliz...
3,city,string,Customer billing city,Required non-empty text
4,category,category,Product family,Learning Kit Course Access or Mentor Session
5,quantity,integer,Units purchased,Whole number greater than zero
6,unit_price,decimal,Price per unit before discount,Non-negative INR amount
7,discount_pct,decimal,Percentage discount applied,Between 0 and 100; missing means zero only aft...
8,payment_status,category,Latest order settlement state,Paid Pending Failed or Refunded after normaliz...


## 1. Completeness
**Rule:** `order_id`, `order_date`, `city` are required fields per the dictionary; `discount_pct`
should only be null when explicitly justified. Threshold: **≥ 99% non-null** for required fields.

In [3]:
required_cols = ['order_id', 'order_date', 'customer_segment', 'city', 'category',
                  'quantity', 'unit_price', 'payment_status']

completeness = pd.DataFrame({
    'non_null_count': raw[required_cols + ['discount_pct']].notna().sum(),
    'total_rows': len(raw),
})
completeness['non_null_pct'] = (completeness['non_null_count'] / completeness['total_rows'] * 100).round(1)
completeness['threshold_pct'] = 99.0
completeness['status'] = np.where(completeness['non_null_pct'] >= completeness['threshold_pct'], 'PASS', 'FAIL')
completeness


,non_null_count,total_rows,non_null_pct,threshold_pct,status
order_id,12,12,100.0,99.0,PASS
order_date,11,12,91.7,99.0,FAIL
customer_segment,12,12,100.0,99.0,PASS
city,11,12,91.7,99.0,FAIL
category,12,12,100.0,99.0,PASS
quantity,12,12,100.0,99.0,PASS
unit_price,12,12,100.0,99.0,PASS
payment_status,12,12,100.0,99.0,PASS
discount_pct,11,12,91.7,99.0,FAIL


In [4]:
# Trace which specific orders have missing required fields
missing_report = []
for col in required_cols:
    missing_ids = raw.loc[raw[col].isna() | (raw[col].astype(str).str.strip() == ''), 'order_id'].tolist()
    if missing_ids:
        missing_report.append({'column': col, 'missing_in_orders': missing_ids})

pd.DataFrame(missing_report)


,column,missing_in_orders
0,order_date,[RT-1011]
1,city,[RT-1005]


## 2. Uniqueness
**Rule:** `order_id` must be unique (dictionary: *"Required and unique"*). Threshold: **0 duplicates.**

In [5]:
dupe_mask = raw.duplicated(subset=['order_id'], keep=False)
dupes = raw.loc[dupe_mask].sort_values('order_id')

n_duplicate_ids = raw['order_id'].duplicated().sum()
status = 'PASS' if n_duplicate_ids == 0 else 'FAIL'
print(f"Duplicate order_id count: {n_duplicate_ids}  ->  {status}")
dupes


Duplicate order_id count: 1  ->  FAIL


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid


## 3. Validity
Executable checks against the exact rules in the data dictionary:

- `order_date`: must parse as an ISO date between 2025-01-01 and today.
- `quantity`: whole number, greater than zero.
- `unit_price`: non-negative INR amount.
- `discount_pct`: between 0 and 100 (when present).
- `category`: one of `Learning Kit`, `Course Access`, `Mentor Session`.
- `customer_segment` / `payment_status`: validity checked after normalization in the Consistency section.

In [6]:
def parse_iso_date(value):
    """Return a pandas Timestamp only if the value is a *strict* ISO (YYYY-MM-DD) date
    within the allowed business range; otherwise return NaT so it fails validity."""
    if pd.isna(value):
        return pd.NaT
    try:
        ts = pd.to_datetime(value, format='%Y-%m-%d', errors='raise')
    except (ValueError, TypeError):
        return pd.NaT
    if ts < pd.Timestamp('2025-01-01') or ts > pd.Timestamp.today().normalize():
        return pd.NaT
    return ts

raw['order_date_parsed'] = raw['order_date'].apply(parse_iso_date)
invalid_dates = raw.loc[raw['order_date'].notna() & raw['order_date_parsed'].isna(), ['order_id', 'order_date']]

valid_pct = round(raw['order_date_parsed'].notna().sum() / len(raw) * 100, 1)
print(f"order_date valid & in-range: {valid_pct}%  (threshold >= 99%)  -> {'PASS' if valid_pct >= 99 else 'FAIL'}")
invalid_dates


order_date valid & in-range: 75.0%  (threshold >= 99%)  -> FAIL


,order_id,order_date
1,RT-1002,03/01/2026
6,RT-1006,2026-13-10


In [7]:
def parse_positive_int(value):
    try:
        v = int(str(value).strip())
        return v if v > 0 else np.nan
    except (ValueError, TypeError):
        return np.nan

raw['quantity_parsed'] = raw['quantity'].apply(parse_positive_int)
invalid_qty = raw.loc[raw['quantity'].notna() & raw['quantity_parsed'].isna(), ['order_id', 'quantity']]

valid_pct = round(raw['quantity_parsed'].notna().sum() / len(raw) * 100, 1)
print(f"quantity valid positive integer: {valid_pct}%  (threshold = 100%)  -> {'PASS' if valid_pct == 100 else 'FAIL'}")
invalid_qty


quantity valid positive integer: 83.3%  (threshold = 100%)  -> FAIL


,order_id,quantity
6,RT-1006,-1
8,RT-1008,two


In [8]:
raw['unit_price_num'] = pd.to_numeric(raw['unit_price'], errors='coerce')
invalid_price = raw.loc[(raw['unit_price_num'].isna()) | (raw['unit_price_num'] < 0), ['order_id', 'unit_price']]
status = 'PASS' if invalid_price.empty else 'FAIL'
print(f"unit_price non-negative numeric: {status}")
invalid_price


unit_price non-negative numeric: PASS


,order_id,unit_price


In [9]:
raw['discount_pct_num'] = pd.to_numeric(raw['discount_pct'], errors='coerce')
present = raw['discount_pct'].notna()
out_of_range = raw.loc[present & ((raw['discount_pct_num'] < 0) | (raw['discount_pct_num'] > 100)), ['order_id', 'discount_pct']]

valid_pct = round((present.sum() - len(out_of_range)) / present.sum() * 100, 1) if present.sum() else 100.0
print(f"discount_pct within 0-100 (of non-null values): {valid_pct}%  (threshold = 100%)  -> {'PASS' if out_of_range.empty else 'FAIL'}")
out_of_range


discount_pct within 0-100 (of non-null values): 90.9%  (threshold = 100%)  -> FAIL


,order_id,discount_pct
7,RT-1007,105


In [10]:
allowed_categories = {'Learning Kit', 'Course Access', 'Mentor Session'}
invalid_cat = raw.loc[~raw['category'].isin(allowed_categories), ['order_id', 'category']]
status = 'PASS' if invalid_cat.empty else 'FAIL'
print(f"category in allowed set {allowed_categories}: {status}")
invalid_cat


category in allowed set {'Course Access', 'Learning Kit', 'Mentor Session'}: PASS


,order_id,category


## 4. Consistency
**Rule:** `customer_segment` must normalize to `{Student, Fresher, Professional}` and
`payment_status` to `{Paid, Pending, Failed, Refunded}` — case and whitespace should not create
phantom categories.

In [11]:
def normalize_categorical(series):
    return series.str.strip().str.title()

raw['customer_segment_norm'] = normalize_categorical(raw['customer_segment'])
raw['payment_status_norm'] = normalize_categorical(raw['payment_status'])

print("customer_segment — raw distinct values :", sorted(raw['customer_segment'].dropna().unique()))
print("customer_segment — normalized values   :", sorted(raw['customer_segment_norm'].dropna().unique()))
print()
print("payment_status — raw distinct values   :", sorted(raw['payment_status'].dropna().unique()))
print("payment_status — normalized values     :", sorted(raw['payment_status_norm'].dropna().unique()))

allowed_segment = {'Student', 'Fresher', 'Professional'}
allowed_payment = {'Paid', 'Pending', 'Failed', 'Refunded'}

seg_ok = set(raw['customer_segment_norm'].dropna().unique()) <= allowed_segment
pay_ok = set(raw['payment_status_norm'].dropna().unique()) <= allowed_payment
print()
print(f"customer_segment consistent after normalization -> {'PASS' if seg_ok else 'FAIL'}")
print(f"payment_status consistent after normalization    -> {'PASS' if pay_ok else 'FAIL'}")


customer_segment — raw distinct values : ['Fresher', 'Professional', 'Student', 'student']
customer_segment — normalized values   : ['Fresher', 'Professional', 'Student']

payment_status — raw distinct values   : ['Failed', 'Paid', 'Pending', 'Refunded', 'paid']
payment_status — normalized values     : ['Failed', 'Paid', 'Pending', 'Refunded']

customer_segment consistent after normalization -> PASS
payment_status consistent after normalization    -> PASS


## 5. Freshness
**Rule:** the pipeline should ingest orders promptly. We measure the gap between the most recent
*valid* `order_date` and the moment this notebook runs. Threshold: **≤ 24 hours** lag at the daily
refresh checkpoint (this dataset is a static sample, so in production this cell would run on a
schedule right after ingestion).

In [12]:
latest_valid_date = raw['order_date_parsed'].max()
now = pd.Timestamp.today()
lag_hours = (now - latest_valid_date).total_seconds() / 3600 if pd.notna(latest_valid_date) else np.nan

print(f"Most recent valid order_date : {latest_valid_date.date() if pd.notna(latest_valid_date) else 'N/A'}")
print(f"Notebook run time            : {now}")
print(f"Freshness lag                : {lag_hours:,.1f} hours")
print("NOTE: this is a static sample file, so lag will typically exceed the 24h production threshold.")
print("In the live pipeline this check runs immediately after each ingestion batch, not on a fixed sample.")


Most recent valid order_date : 2026-01-18
Notebook run time            : 2026-09-12 06:26:23.948641
Freshness lag                : 5,694.4 hours
NOTE: this is a static sample file, so lag will typically exceed the 24h production threshold.
In the live pipeline this check runs immediately after each ingestion batch, not on a fixed sample.


## 6. Summary Scorecard
Rolls every dimension up into a single pass/fail table — this is what should feed the Data
Quality Contract's escalation logic.

In [13]:
scorecard = pd.DataFrame([
    {'dimension': 'Completeness', 'check': 'Required fields >= 99% non-null',
     'result': f"{completeness['non_null_pct'].min()}% (worst column)",
     'status': 'PASS' if completeness['non_null_pct'].min() >= 99 else 'FAIL'},
    {'dimension': 'Uniqueness', 'check': 'order_id has zero duplicates',
     'result': f"{n_duplicate_ids} duplicate(s)",
     'status': 'PASS' if n_duplicate_ids == 0 else 'FAIL'},
    {'dimension': 'Validity', 'check': 'order_date parses & in range',
     'result': f"{round(raw['order_date_parsed'].notna().sum() / len(raw) * 100, 1)}%",
     'status': 'PASS' if raw['order_date_parsed'].notna().sum() / len(raw) >= 0.99 else 'FAIL'},
    {'dimension': 'Validity', 'check': 'quantity positive integer',
     'result': f"{round(raw['quantity_parsed'].notna().sum() / len(raw) * 100, 1)}%",
     'status': 'PASS' if raw['quantity_parsed'].notna().sum() == len(raw) else 'FAIL'},
    {'dimension': 'Validity', 'check': 'discount_pct within 0-100',
     'result': f"{len(out_of_range)} row(s) out of range",
     'status': 'PASS' if out_of_range.empty else 'FAIL'},
    {'dimension': 'Validity', 'check': 'category in allowed set',
     'result': f"{len(invalid_cat)} invalid row(s)",
     'status': 'PASS' if invalid_cat.empty else 'FAIL'},
    {'dimension': 'Consistency', 'check': 'customer_segment normalizes cleanly',
     'result': 'OK' if seg_ok else 'Unexpected values found',
     'status': 'PASS' if seg_ok else 'FAIL'},
    {'dimension': 'Consistency', 'check': 'payment_status normalizes cleanly',
     'result': 'OK' if pay_ok else 'Unexpected values found',
     'status': 'PASS' if pay_ok else 'FAIL'},
    {'dimension': 'Freshness', 'check': 'Lag from latest valid order_date <= 24h',
     'result': f"{lag_hours:,.1f} hours" if pd.notna(lag_hours) else 'N/A',
     'status': 'PASS' if pd.notna(lag_hours) and lag_hours <= 24 else 'FAIL (static sample file)'},
])
scorecard


,dimension,check,result,status
0,Completeness,Required fields >= 99% non-null,91.7% (worst column),FAIL
1,Uniqueness,order_id has zero duplicates,1 duplicate(s),FAIL
2,Validity,order_date parses & in range,75.0%,FAIL
3,Validity,quantity positive integer,83.3%,FAIL
4,Validity,discount_pct within 0-100,1 row(s) out of range,FAIL
5,Validity,category in allowed set,0 invalid row(s),PASS
6,Consistency,customer_segment normalizes cleanly,OK,PASS
7,Consistency,payment_status normalizes cleanly,OK,PASS
8,Freshness,Lag from latest valid order_date <= 24h,"5,694.4 hours",FAIL (static sample file)


## 7. Clean, Analysis-Ready Extract
Applies every fix implied by the checks above (normalize categories, reformat dates, drop
duplicates, drop unrecoverable rows) so downstream KPI formulas in the KPI Dictionary can run
against trustworthy data. Rejected rows are kept in a separate quarantine table for the ops team
to correct at source, per the escalation actions in the Data Quality Contract.

In [14]:
clean = raw.copy()
clean['customer_segment'] = clean['customer_segment_norm']
clean['payment_status'] = clean['payment_status_norm']
clean['order_date'] = clean['order_date_parsed']
clean['quantity'] = clean['quantity_parsed']
clean['unit_price'] = clean['unit_price_num']
clean['discount_pct'] = clean['discount_pct_num']

# Drop the temporary helper columns
clean = clean.drop(columns=[c for c in clean.columns if c.endswith(('_norm', '_parsed', '_num'))])

# 1) De-duplicate on order_id (keep first occurrence)
before = len(clean)
clean = clean.drop_duplicates(subset=['order_id'], keep='first')
print(f"Removed {before - len(clean)} exact duplicate order_id row(s)")

# 2) Quarantine rows that fail hard validity/completeness rules
hard_fail_mask = (
    clean['order_date'].isna() |
    clean['city'].isna() | (clean['city'].astype(str).str.strip() == '') |
    clean['quantity'].isna() |
    clean['unit_price'].isna()
)
quarantine = clean.loc[hard_fail_mask].copy()
clean = clean.loc[~hard_fail_mask].copy()

print(f"Quarantined {len(quarantine)} row(s) failing hard validity/completeness rules")
print(f"Analysis-ready rows remaining: {len(clean)} of {len(raw)}")

print("\n--- Quarantined rows (route to source-system owner) ---")
display(quarantine[['order_id', 'order_date', 'city', 'quantity', 'unit_price']])

print("\n--- Clean, analysis-ready extract ---")
clean


Removed 1 exact duplicate order_id row(s)
Quarantined 5 row(s) failing hard validity/completeness rules
Analysis-ready rows remaining: 6 of 12

--- Quarantined rows (route to source-system owner) ---


,order_id,order_date,city,quantity,unit_price
1,RT-1002,NaT,Bengaluru,1.0,1499
5,RT-1005,2026-01-09,NaN,1.0,999
6,RT-1006,NaT,Pune,NaN,799
8,RT-1008,2026-01-14,Bengaluru,NaN,1499
11,RT-1011,NaT,Kochi,1.0,999



--- Clean, analysis-ready extract ---


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2.0,799,10.0,Paid
2,RT-1003,2026-01-05,Student,Chennai,Course Access,1.0,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3.0,799,5.0,Paid
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2.0,999,105.0,Paid
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1.0,799,0.0,Paid
10,RT-1010,2026-01-18,Professional,Delhi,Course Access,2.0,1499,15.0,Refunded


In [15]:
# Persist outputs so they can be handed to BI / the KPI workbook
clean.to_csv('retail_orders_clean.csv', index=False)
quarantine.to_csv('retail_orders_quarantine.csv', index=False)
scorecard.to_csv('data_quality_scorecard.csv', index=False)
print('Wrote: retail_orders_clean.csv, retail_orders_quarantine.csv, data_quality_scorecard.csv')


Wrote: retail_orders_clean.csv, retail_orders_quarantine.csv, data_quality_scorecard.csv
